#Ask Your Data Assistant

**Domain:** BI / Text-to-SQL AI

**Dataset:** Olist Brazilian E-Commerce -- https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce

**Skills:** Text-to-SQL, LLM Integration, Database Interaction, Prompt Design, AI Safety & Validation

**Tools:** Python, SQLite, Cerebras / Groq / Ollama (free tier, via shared call_llm()), Pandas

---

### Problem Statement
Business users want to ask questions about company data in plain English instead of learning SQL or waiting on the analytics team. They need a natural-language interface that safely generates and executes SQL, then explains the answer in plain English.

In [1]:
!pip install -q kagglehub openai

### Setup

In [2]:
import sqlite3
import pandas as pd
import numpy as np
# ---- AI setup: same pattern for every project, so it's one thing to remember ----
# we try cerebras first (biggest free daily limit), then groq, then a local ollama model
# if all 3 are down for some reason we just return a message instead of crashing the whole script
import os
from getpass import getpass
import requests
from openai import OpenAI
import warnings
warnings.filterwarnings("ignore")

# asking for api keys only if they are not already set, and using getpass so the key
# does not get printed on screen or saved inside the notebook by accident
if not os.environ.get("CEREBRAS_API_KEY"):
    os.environ["CEREBRAS_API_KEY"] = getpass("Enter your Cerebras API key: ")
if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ")

cerebras_client = OpenAI(api_key=os.environ["CEREBRAS_API_KEY"], base_url="https://api.cerebras.ai/v1")
groq_client = OpenAI(api_key=os.environ["GROQ_API_KEY"], base_url="https://api.groq.com/openai/v1")
OLLAMA_URL = "http://localhost:11434/api/generate"

def call_llm(prompt):
    # step 1: try cerebras, it has the biggest free tier (1M tokens/day)
    try:
        response = cerebras_client.chat.completions.create(
            model="gpt-oss-120b",
            messages=[{"role": "user", "content": prompt}],
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print("cerebras failed:", e)

    # step 2: cerebras down or key wrong, try groq next
    try:
        response = groq_client.chat.completions.create(
            model="openai/gpt-oss-120b",
            messages=[{"role": "user", "content": prompt}],
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print("groq failed:", e)

    # step 3: last resort, try a local ollama model if one happens to be running
    try:
        resp = requests.post(OLLAMA_URL, json={"model": "llama3", "prompt": prompt, "stream": False})
        resp.raise_for_status()
        return resp.json()["response"].strip()
    except Exception as e:
        print("ollama failed too:", e)
        return "AI call failed, all 3 options did not work"

# ---- dataset loading helper: tries kaggle first, never crashes the notebook ----
# kagglehub comes pre-installed on google colab, but just in case it's missing
# (running locally, older colab image etc) we try to pip install it quietly first
try:
    import kagglehub
except ImportError:
    import subprocess
    subprocess.run(["pip", "install", "-q", "kagglehub", "--break-system-packages"], check=False)
    try:
        import kagglehub
    except ImportError:
        kagglehub = None  # still not available, load_dataset_safe below will just use fake data

# a lot of the kaggle datasets in this project series randomly fail to download
# (auth issues, dataset moved, rate limit, no internet on the runtime, kagglehub missing etc)
# so instead of letting the whole notebook die, we fall back to a small fake dataset
# that has the exact same columns, so every line of code below still works
def load_dataset_safe(kaggle_handle, csv_name, synthetic_fn, n_rows=1500):
    if kagglehub is None:
        print("kagglehub isn't available, using synthetic sample data instead so the rest of the code still runs")
        return synthetic_fn(n_rows)
    try:
        path = kagglehub.dataset_download(kaggle_handle)
        full_path = os.path.join(path, csv_name)
        df = pd.read_csv(full_path)
        print(f"loaded real dataset from kaggle, {len(df)} rows")
        return df
    except Exception as e:
        print("could not download from kaggle (", e, ") - using synthetic sample data instead so the rest of the code still runs")
        return synthetic_fn(n_rows)


def make_fake_orders(n_rows=1500):
    np.random.seed(2)
    return pd.DataFrame({
        'order_id': range(1, n_rows + 1), 'customer_id': np.random.randint(1, 400, n_rows),
        'order_status': np.random.choice(['delivered', 'shipped', 'canceled'], n_rows, p=[0.8, 0.15, 0.05]),
        'order_purchase_timestamp': pd.date_range('2023-01-01', periods=n_rows, freq='h'),
    })

def make_fake_order_items(n_rows=2000):
    np.random.seed(12)
    return pd.DataFrame({
        'order_id': np.random.randint(1, 1500, n_rows), 'product_id': np.random.randint(1, 300, n_rows),
        'price': np.round(np.random.uniform(10, 400, n_rows), 2),
        'freight_value': np.round(np.random.uniform(2, 40, n_rows), 2),
    })

def make_fake_products(n_rows=300):
    np.random.seed(13)
    cats = ['electronics', 'furniture', 'toys', 'sports', 'books']
    return pd.DataFrame({'product_id': range(1, n_rows + 1), 'product_category_name': np.random.choice(cats, n_rows)})

Enter your Cerebras API key: ··········
Enter your Groq API key: ··········


### Step 1: Load Data Into Sqlite So The Assistant Has Something Real To Query

In [3]:
# 1. LOAD DATA INTO SQLITE SO THE ASSISTANT HAS SOMETHING REAL TO QUERY
conn = sqlite3.connect('ecommerce.db')

orders = load_dataset_safe('olistbr/brazilian-ecommerce', 'olist_orders_dataset.csv', make_fake_orders, n_rows=1500)
order_items = load_dataset_safe('olistbr/brazilian-ecommerce', 'olist_order_items_dataset.csv', make_fake_order_items, n_rows=2000)
products = load_dataset_safe('olistbr/brazilian-ecommerce', 'olist_products_dataset.csv', make_fake_products, n_rows=300)

orders.to_sql('orders', conn, if_exists='replace', index=False)
order_items.to_sql('order_items', conn, if_exists='replace', index=False)
products.to_sql('products', conn, if_exists='replace', index=False)

Using Colab cache for faster access to the 'brazilian-ecommerce' dataset.
loaded real dataset from kaggle, 99441 rows
Using Colab cache for faster access to the 'brazilian-ecommerce' dataset.
loaded real dataset from kaggle, 112650 rows
Using Colab cache for faster access to the 'brazilian-ecommerce' dataset.
loaded real dataset from kaggle, 32951 rows


32951

### Step 2: Give The Llm The Schema So It Knows What Columns Exist

In [4]:
# 2. GIVE THE LLM THE SCHEMA SO IT KNOWS WHAT COLUMNS EXIST (it can't guess this correctly otherwise)
def get_schema_text(conn, tables):
    schema_parts = []
    for t in tables:
        cols = pd.read_sql(f"PRAGMA table_info({t})", conn)['name'].tolist()
        schema_parts.append(f"Table {t}: columns = {', '.join(cols)}")
    return "\n".join(schema_parts)

schema_text = get_schema_text(conn, ['orders', 'order_items', 'products'])

### Step 3: Ask The Llm To Turn A Question Into Sql

In [5]:
# 3. ASK THE LLM TO TURN A QUESTION INTO SQL
def generate_sql(question, schema_text):
    prompt = f"""You are a SQLite expert. Given this schema:
{schema_text}

Write ONE read-only SQLite SELECT query to answer the question.
Return ONLY the SQL query, no explanation, no markdown.

Question: {question}
SQL:"""
    sql = call_llm(prompt)
    return sql.replace('```sql', '').replace('```', '').strip()

### Step 4: Never Run A Query Blindly -- Check It First

In [6]:
# 4. NEVER RUN A QUERY BLINDLY -- CHECK IT FIRST
BLOCKED_KEYWORDS = ['DROP', 'DELETE', 'UPDATE', 'INSERT', 'ALTER', 'TRUNCATE', 'CREATE']

def is_sql_safe(sql):
    sql_upper = sql.upper()
    if not sql_upper.strip().startswith('SELECT'):
        return False
    return not any(kw in sql_upper for kw in BLOCKED_KEYWORDS)

### Step 5: Turn The Raw Result Table Into A Plain English Answer

In [7]:
# 5. TURN THE RAW RESULT TABLE INTO A PLAIN ENGLISH ANSWER
def explain_result(question, result_df):
    prompt = f"""The user asked: "{question}"
The SQL query returned this result:
{result_df.to_string(index=False)}

Answer the user's question in one clear, plain-English sentence using these numbers."""
    return call_llm(prompt)

def ask_data_assistant(question):
    sql = generate_sql(question, schema_text)
    if not is_sql_safe(sql):
        return {"error": "generated SQL failed the safety check", "sql": sql}
    try:
        result_df = pd.read_sql(sql, conn)
    except Exception as e:
        return {"error": str(e), "sql": sql}
    answer = explain_result(question, result_df.head(20))
    return {"sql": sql, "result": result_df, "answer": answer}

# --- TESTING WITH 5 QUESTIONS ---
questions = [
    "What was the total number of orders?",
    "Which product category has the most items sold?",
    "What is the average freight value per order?",
    "How many orders were canceled?",
    "What is the total price of all order items?",
]
for q in questions:
    result = ask_data_assistant(q)
    print(f"Q: {q}\nSQL: {result.get('sql')}\nA: {result.get('answer')}\n{'-'*60}")

cerebras failed: Error code: 402 - {'message': 'Payment required to access this resource. Visit your billing tab.', 'type': 'payment_required_error', 'param': 'quota', 'code': 'payment_required'}
cerebras failed: Error code: 402 - {'message': 'Payment required to access this resource. Visit your billing tab.', 'type': 'payment_required_error', 'param': 'quota', 'code': 'payment_required'}
Q: What was the total number of orders?
SQL: SELECT COUNT(*) AS total_orders FROM orders;
A: The total number of orders was **99,441**.
------------------------------------------------------------
cerebras failed: Error code: 402 - {'message': 'Payment required to access this resource. Visit your billing tab.', 'type': 'payment_required_error', 'param': 'quota', 'code': 'payment_required'}
cerebras failed: Error code: 402 - {'message': 'Payment required to access this resource. Visit your billing tab.', 'type': 'payment_required_error', 'param': 'quota', 'code': 'payment_required'}
Q: Which product ca